# CropMedic Batch 1 — Common Bean pilot

This notebook trains the **audited BLD-BD Common Bean pilot dataset** prepared locally with CropMedic.

Expected classes: **Angular Leaf Spot, Bean Common Mosaic Virus, Healthy** — 224 original images per class, 672 total.

This remains a pilot model. Do **not** mark Common bean as AI-supported until independent field validation and rejection testing pass.

In [ ]:
!nvidia-smi
!git clone -b production-v1 https://github.com/macdonaldchigutiro/CropMedic.git
%cd CropMedic/crop_medic_backend/training
!pip -q install -r requirements-training.txt

## Upload the audited dataset ZIP

On the Windows PC, first create `common_bean_for_colab.zip` from the already-audited `data_batch1/common_bean` class folders. Then run the next cell and choose that ZIP when Colab asks for a file.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, zipfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError('Upload exactly one ZIP: common_bean_for_colab.zip')
zip_name = next(iter(uploaded))
if not zip_name.lower().endswith('.zip'):
    raise RuntimeError('The uploaded file must be a ZIP archive.')

target = Path('data_batch1/common_bean')
if target.exists():
    shutil.rmtree(target)
target.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall(target)
print('Extracted to', target.resolve())

In [ ]:
!python audit_crop_dataset.py --data-dir data_batch1/common_bean --crop-id common_bean

from pathlib import Path
classes = sorted([p.name for p in Path('data_batch1/common_bean').iterdir() if p.is_dir()])
image_count = sum(1 for p in Path('data_batch1/common_bean').rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'})
print('Classes:', classes)
print('Images:', image_count)
expected = ['Angular Leaf Spot', 'Bean Common Mosaic Virus', 'Healthy']
if classes != expected or image_count != 672:
    raise RuntimeError('Dataset does not match the audited 672-image Common Bean pilot. Stop before training.')


In [ ]:
!python train_crop_model.py --crop-id common_bean --data-dir data_batch1/common_bean --output-dir artifacts_batch1/common_bean --head-epochs 10 --fine-epochs 18

In [ ]:
import json
m = json.load(open('artifacts_batch1/common_bean/metrics.json'))
print('TEST ACCURACY:', m['test_accuracy'])
print('MACRO F1:', m['macro_f1'])
print(json.dumps(m['per_class'], indent=2))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/CropMedic/batch1/common_bean
!cp -r artifacts_batch1/common_bean/* /content/drive/MyDrive/CropMedic/batch1/common_bean/
print('Saved to Google Drive: MyDrive/CropMedic/batch1/common_bean')

## What happens after training

Freeze these pilot metrics before using a separate field dataset for external validation. Do not tune on the external validation set. Common bean remains **Coming soon** in the app until field validation and uncertainty/rejection checks pass.